# 07 · Express Deployment

## Express Deployments 란

기존 Model Serving 의 endpoint 빌드 = **container build** 단계 포함, 보통 10분 이상.

Express Deployments 는 **모델 등록 시점에** serverless 노트북 환경 + 모델 아티팩트를
pre-stage 해두고, endpoint 생성 시 그걸 재사용 → **container build 단계 스킵**.

> Endpoint READY 시간: ~10분 → **1-2분**

## Prerequisites

- **Serverless Notebook v3 또는 v4** 에서 logging + registration (이 노트북도 serverless 권장)
- `mlflow >= 3.1`
- Unity Catalog 등록
- **CPU serving** (GPU 미지원)
- Environment size ≤ 1 GB
- Custom model (Foundation Model APIs 아님)

In [ ]:
%pip install -q "mlflow>=3.1.0" "databricks-sdk>=0.30.0"
%restart_python

In [ ]:
%run ./config

In [ ]:
import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

## Step 1. 가벼운 모델 학습 (Serverless 환경 그대로 사용)

Express 의 핵심은 **현재 노트북 환경을 그대로 서빙에 사용**한다는 점.
무거운 라이브러리는 추가하지 마세요 (1 GB cap).

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from mlflow.models import infer_signature

pdf = spark.table(f"{catalog}.{schema}.customers").toPandas()
FEATURES = ["age", "tenure_months", "monthly_charges", "total_charges", "support_tickets"]
X_train, X_test, y_train, y_test = train_test_split(
    pdf[FEATURES], pdf["churned"], test_size=0.2, random_state=42
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr",     LogisticRegression(max_iter=500, random_state=42)),
])
pipe.fit(X_train, y_train)
print(f"test accuracy = {pipe.score(X_test, y_test):.3f}")

## Step 2. Log + Register with `EnvPackConfig`

`env_pack=EnvPackConfig(name="databricks_model_serving")` 가 핵심.
이게 있으면 serverless env 가 serving 용으로 staging 됩니다.

In [ ]:
sig = infer_signature(X_train, pipe.predict(X_train))

with mlflow.start_run(run_name="express_lr") as run:
    info = mlflow.sklearn.log_model(
        sk_model=pipe,
        name="model",
        signature=sig,
        input_example=X_train.iloc[:5],
    )
    print(f"logged: {info.model_uri}")

### Express 모드로 등록

`EnvPackConfig` 가 import 안 되는 경우 — MLflow 3.1 미만이거나 plugin 미설치. 그 경우엔 standard registration 후 일반 endpoint 생성 (=06번 노트북).

In [ ]:
try:
    from mlflow.utils.env_pack import EnvPackConfig

    mv = mlflow.register_model(
        model_uri=info.model_uri,
        name=model_express,
        env_pack=EnvPackConfig(name="databricks_model_serving"),
    )
    print(f"✓ Express registration: {model_express} v{mv.version}")
    express_available = True

except (ImportError, ValueError) as e:
    print(f"⚠ EnvPackConfig 미지원 또는 비serverless 환경 — standard registration 으로 fallback ({e})")
    mv = mlflow.register_model(model_uri=info.model_uri, name=model_express)
    express_available = False

## Step 3. UI 에서 원클릭 배포

1. 사이드바 → **Catalog** → `main` → `model_serving_cookbook` → `churn_express`
2. **"Use model for inference"** → **"Serving endpoint"**
3. Defaults: workload_type **CPU**, workload_size **Small**, scale_to_zero **enabled**
4. **Create** — Express 라면 ~1-2분 내 READY

## Step 4. SDK 로 endpoint 생성 (UI 와 동일한 결과)

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput, TrafficConfig, Route,
)
import time

w = WorkspaceClient()
served_name = f"express-v{mv.version}"

t0 = time.time()
endpoint = w.serving_endpoints.create_and_wait(
    name=endpoint_express,
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                name=served_name,
                entity_name=model_express,
                entity_version=str(mv.version),
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ],
        traffic_config=TrafficConfig(
            routes=[Route(served_model_name=served_name, traffic_percentage=100)]
        ),
    ),
    timeout=__import__("datetime").timedelta(minutes=15),
)
elapsed = time.time() - t0
print(f"✓ {endpoint_express} READY in {elapsed/60:.1f} 분")
print(f"   (Express 활성 시 1-2분, 일반 빌드 시 5-10분)")

## Step 5. 호출 테스트

In [ ]:
import mlflow.deployments
deploy_client = mlflow.deployments.get_deploy_client("databricks")
sample = X_test.iloc[:3]
resp = deploy_client.predict(
    endpoint=endpoint_express,
    inputs={"dataframe_split": {"columns": sample.columns.tolist(), "data": sample.values.tolist()}},
)
print(resp)

## Step 6. Scale-to-zero 비용 프로파일

`scale_to_zero_enabled=True` 시:

| 상태 | 비용 |
| --- | --- |
| Idle (no requests) | **$0** — 0 replica |
| First request after idle | Cold start: Express 시 ~30s, 일반 CPU ~60s |
| Warm (active) | Replica-hour 당 CPU Model Serving DBU rate |

**프로덕션 SLA 비권장** (scaled-to-zero 일 때 capacity 보장 X). 데모 / 내부 도구 / 저빈도 호출에 최적.

### Provisioned vs Scale-to-zero

| 시나리오 | 권장 |
| --- | --- |
| 사내 데모 / dev / test | scale_to_zero **on** |
| 24/7 production with SLA | scale_to_zero **off**, provisioned (`workload_size` Medium+) |
| FMAPI / 고QPS LLM | Provisioned throughput (별도 ramp) |

## 정리

| 기능 | 효과 |
| --- | --- |
| `EnvPackConfig(name="databricks_model_serving")` | container build 스킵 → endpoint READY 10분 → 1-2분 |
| scale_to_zero | idle 시 $0 |
| UC 모델 페이지 "Use model for inference" | UI 1-click 배포 |
| MLflow 3 Deployment Jobs | 등록 → endpoint 자동화 (one-shot 파이프라인에 근접) |

## 피치 (한 줄)

> **"Serverless 노트북에서 학습 → `env_pack` 으로 등록 → UC 에서 한 번 클릭 → 1분 안에 CPU 엔드포인트, 유휴 시 $0."**

→ 다음: **`99_cleanup`** — 데모 후 리소스 정리